# argentina.educacion — Pruebas interactivas

Recorrido paso a paso del módulo `argentina.educacion`.

Funciones simples para limpiar/validar identificadores escolares (CUE, CUEANEXO) y normalizar categorías educativas (sector, ámbito, nivel). Solo stdlib — sin APIs externas, sin padrones, sin pandas, sin scraping.

## 1. Setup e imports

In [1]:
import argentina as arg

print(f"argentina v{arg.__version__}")
print(f"jurisdicciones cargadas: {len(arg.educacion.JURISDICCIONES)}")

argentina v0.0.14
jurisdicciones cargadas: 24


## 2. CUE

**Clave Única de Establecimiento**: 9 dígitos. `limpiar_cue` saca separadores y rellena con ceros a la izquierda. `validar_cue` chequea que los dígitos crudos (antes del padding) sean exactamente 9.

In [2]:
# Caso típico: CUE con guion separador
arg.educacion.limpiar_cue("0201234-00")

'020123400'

In [3]:
# Otros separadores también se sacan
print(arg.educacion.limpiar_cue("02 0123400"))
print(arg.educacion.limpiar_cue("02.0123400"))
print(arg.educacion.limpiar_cue("020123400"))

020123400
020123400
020123400


In [4]:
# Si vienen menos de 9 dígitos, se rellena con ceros — útil cuando un CSV mal formateado los perdió
arg.educacion.limpiar_cue("123")

'000000123'

In [5]:
# Casos sin dígitos → None
print(arg.educacion.limpiar_cue(None))
print(arg.educacion.limpiar_cue(""))
print(arg.educacion.limpiar_cue("abc"))

None
None
None


In [6]:
# Validación: exige 9 dígitos crudos (no acepta el padding implícito)
print(arg.educacion.validar_cue("020123400"))    # 9 dígitos exactos
print(arg.educacion.validar_cue("0201234-00"))   # 9 dígitos con separador
print(arg.educacion.validar_cue("123"))          # corto
print(arg.educacion.validar_cue("1234567890"))   # largo
print(arg.educacion.validar_cue(None))

True
True
False
False
False


## 3. CUEANEXO

El **CUEANEXO** identifica un anexo del establecimiento. Misma estructura que el CUE (9 dígitos), donde los últimos 2 representan el número de anexo (`00` = sede principal).

In [7]:
print(arg.educacion.limpiar_cueanexo("0201234-01"))   # anexo 01
print(arg.educacion.limpiar_cueanexo("0201234-00"))   # sede

020123401
020123400


In [8]:
print(arg.educacion.validar_cueanexo("020123401"))
print(arg.educacion.validar_cueanexo("123"))
print(arg.educacion.validar_cueanexo(None))

True
False
False


## 4. Jurisdicción desde el CUE

Los **2 primeros dígitos del CUE** son el código INDEC de la jurisdicción (mismo esquema que provincias).

In [9]:
arg.educacion.extraer_jurisdiccion_cue("020123400")

'Ciudad Autónoma de Buenos Aires'

In [10]:
# Funciona igual con separadores
for cue in ["0201234-00", "140000000", "82-1234567", "940000001"]:
    print(f"{cue:12} → {arg.educacion.extraer_jurisdiccion_cue(cue)}")

0201234-00   → Ciudad Autónoma de Buenos Aires
140000000    → Córdoba
82-1234567   → Santa Fe
940000001    → Tierra del Fuego


In [11]:
# Códigos que no existen → None
print(arg.educacion.extraer_jurisdiccion_cue("990000000"))
print(arg.educacion.extraer_jurisdiccion_cue(None))

None
None


In [12]:
# Tabla completa de jurisdicciones (24)
for codigo, nombre in sorted(arg.educacion.JURISDICCIONES.items()):
    print(f"{codigo}  {nombre}")

02  Ciudad Autónoma de Buenos Aires
06  Buenos Aires
10  Catamarca
14  Córdoba
18  Corrientes
22  Chaco
26  Chubut
30  Entre Ríos
34  Formosa
38  Jujuy
42  La Pampa
46  La Rioja
50  Mendoza
54  Misiones
58  Neuquén
62  Río Negro
66  Salta
70  San Juan
74  San Luis
78  Santa Cruz
82  Santa Fe
86  Santiago del Estero
90  Tucumán
94  Tierra del Fuego


## 5. normalizar_sector

Mapea variantes comunes (`estatal`/`público`/`publico`/`privado`/`privada`) al label canónico.

In [13]:
for valor in ["estatal", "público", "publico", "PRIVADO", " Privada "]:
    print(f"{valor!r:15} → {arg.educacion.normalizar_sector(valor)!r}")

'estatal'       → 'Estatal'
'público'       → 'Estatal'
'publico'       → 'Estatal'
'PRIVADO'       → 'Privado'
' Privada '     → 'Privado'


In [14]:
# Sin match → None (no inventa)
print(arg.educacion.normalizar_sector("otro"))
print(arg.educacion.normalizar_sector(None))
print(arg.educacion.normalizar_sector(""))

None
None
None


## 6. normalizar_ambito

Solo dos valores canónicos: `Urbano` / `Rural`.

In [15]:
for valor in ["urbano", "URBANO", " rural ", "semi-urbano"]:
    print(f"{valor!r:15} → {arg.educacion.normalizar_ambito(valor)!r}")

'urbano'        → 'Urbano'
'URBANO'        → 'Urbano'
' rural '       → 'Rural'
'semi-urbano'   → None


## 7. normalizar_nivel

Mapea `inicial` / `jardín` / `primaria` / `primario` / `secundaria` / `secundario` / `superior` a 4 labels canónicos.

In [16]:
for valor in ["inicial", "jardín", "jardin", "primario", "PRIMARIA", "secundario", "secundaria", "superior"]:
    print(f"{valor!r:15} → {arg.educacion.normalizar_nivel(valor)!r}")

'inicial'       → 'Inicial'
'jardín'        → 'Inicial'
'jardin'        → 'Inicial'
'primario'      → 'Primaria'
'PRIMARIA'      → 'Primaria'
'secundario'    → 'Secundaria'
'secundaria'    → 'Secundaria'
'superior'      → 'Superior'


In [17]:
# Variantes no soportadas → None (universitario, terciario, etc. quedan fuera por ahora)
print(arg.educacion.normalizar_nivel("universitario"))
print(arg.educacion.normalizar_nivel("terciario"))

None
None


## 8. Combinando todo

Pipeline típico: una fila cruda de un padrón con CUE, sector, ámbito y nivel — limpiar y normalizar todo de una.

In [18]:
registros = [
    {"cue": "0201234-00", "sector": "estatal",  "ambito": "urbano", "nivel": "primaria"},
    {"cue": "140000000",  "sector": "PRIVADO",  "ambito": "rural",  "nivel": "secundario"},
    {"cue": "82-1234567", "sector": "público",  "ambito": "urbano", "nivel": "jardín"},
    {"cue": "abc",        "sector": "otro",     "ambito": "",       "nivel": "universitario"},
]

for r in registros:
    print({
        "cue_valido":    arg.educacion.validar_cue(r["cue"]),
        "cue_limpio":    arg.educacion.limpiar_cue(r["cue"]),
        "jurisdiccion":  arg.educacion.extraer_jurisdiccion_cue(r["cue"]),
        "sector":        arg.educacion.normalizar_sector(r["sector"]),
        "ambito":        arg.educacion.normalizar_ambito(r["ambito"]),
        "nivel":         arg.educacion.normalizar_nivel(r["nivel"]),
    })

{'cue_valido': True, 'cue_limpio': '020123400', 'jurisdiccion': 'Ciudad Autónoma de Buenos Aires', 'sector': 'Estatal', 'ambito': 'Urbano', 'nivel': 'Primaria'}
{'cue_valido': True, 'cue_limpio': '140000000', 'jurisdiccion': 'Córdoba', 'sector': 'Privado', 'ambito': 'Rural', 'nivel': 'Secundaria'}
{'cue_valido': True, 'cue_limpio': '821234567', 'jurisdiccion': 'Santa Fe', 'sector': 'Estatal', 'ambito': 'Urbano', 'nivel': 'Inicial'}
{'cue_valido': False, 'cue_limpio': None, 'jurisdiccion': None, 'sector': None, 'ambito': None, 'nivel': None}


## 9. Tests automáticos

Los tests viven en `tests/test_educacion.py`. Para correrlos:

```bash
cd /Users/tobiasyatche/argentina
pytest tests/test_educacion.py -v
```

## Notas sueltas / TODOs

- Solo stdlib (`re`). Sin pandas, sin requests, sin APIs externas, sin padrones embebidos.
- `validar_cue` chequea formato (9 dígitos crudos), **no** que el establecimiento exista en el padrón oficial. Para eso hace falta un dataset, que queda fuera del scope de este módulo.
- `limpiar_cue` zfillea a 9 cuando el dato viene corto (típico bug de Excel/CSV que come ceros a la izquierda). Si querés rechazar inputs cortos, usá `validar_cue` antes.
- `normalizar_nivel` cubre niveles obligatorios (Inicial / Primaria / Secundaria / Superior). Si en algún momento se quiere distinguir entre superior universitario y no universitario, agregar otro mapping aparte.
- Detección de niveles más finos (modalidad, orientación, ciclo, etc.) queda fuera por ahora.